# Desplazamiento ICT — Entrenamiento Multi-Configuración con GPU + CPU Paralelo (≥10 CPUs)

**Objetivo:** Evaluar 8 configuraciones de hiperparámetros usando GPU para entrenamiento y CPU paralelo (≥10 núcleos) para coordinación y ejecución concurrente.

**Hardware objetivo:** Kaggle GPU T4 (para entrenamiento) + CPU paralelo (para gestión de configs)

**Estrategia:** GPU = entrenamiento rápido; CPU ≥10 núcleos = ejecutar varios entrenamientos en paralelo o fallback a CPU pura

**Dataset:** M5 etiquetado (24,980 velas) con 33 features (24 base + 9 HTF)

**Agentes:** Forge (GPU), Probe (8 configs), Sentinel (validación), Vigil (early-stop), Nexus (poda)

In [ ]:
import os, json, time, multiprocessing as mp
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from pathlib import Path

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

print('=== Hardware Check ===')
print(f'TF: {tf.__version__}')

# CPU count
cpu_count = mp.cpu_count()
print(f'CPUs disponibles: {cpu_count}')

# GPU check
gpus = tf.config.list_physical_devices('GPU')
print(f'GPUs: {len(gpus)}')
for i, g in enumerate(gpus):
    print(f'  GPU {i}: {g}')

# Configurar número de workers según hardware
if len(gpus) > 0:
    # Con GPU: usamos GPU para entrenamiento, CPU para data loading
    # Limitamos CPU threads para no competir con GPU
    tf.config.threading.set_inter_op_parallelism_threads(2)
    tf.config.threading.set_intra_op_parallelism_threads(4)
    NUM_WORKERS = max(10, cpu_count) if cpu_count >= 10 else cpu_count
    print(f'GPU disponible → usando {NUM_WORKERS} CPUs para data loading + batch prefetech')
else:
    # Sin GPU: usar CPU paralelo con multiprocessing
    NUM_WORKERS = max(10, cpu_count) if cpu_count >= 10 else cpu_count
    print(f'SIN GPU → usando {NUM_WORKERS} CPUs en paralelo para entrenamiento')

print(f'Workers configurados: {NUM_WORKERS}')

# Verificar que se pueden usar al menos 10 CPUs
if NUM_WORKERS < 10:
    print(f'⚠️  Advertencia: solo hay {NUM_WORKERS} CPUs disponibles (mínimo 10 solicitado)')
else:
    print(f'✓ Se usarán {NUM_WORKERS} CPUs en paralelo (cumple mínimo de 10)')

In [ ]:
# === CONFIGURACIONES (8 en lugar de 32 — diseño experimental de PROBE) ===
CONFIGS = [
    {'name': 'baseline',    'lr': 0.001, 'batch_size': 256, 'dropout': 0.2, 'hidden': 32, 'epochs': 30},
    {'name': 'lr_low',      'lr': 0.0005, 'batch_size': 256, 'dropout': 0.2, 'hidden': 32, 'epochs': 30},
    {'name': 'batch_large', 'lr': 0.001, 'batch_size': 512, 'dropout': 0.2, 'hidden': 32, 'epochs': 30},
    {'name': 'dropout_hi',  'lr': 0.001, 'batch_size': 256, 'dropout': 0.4, 'hidden': 32, 'epochs': 30},
    {'name': 'hidden_hi',   'lr': 0.001, 'batch_size': 256, 'dropout': 0.2, 'hidden': 64, 'epochs': 30},
    {'name': 'conservative','lr': 0.0005, 'batch_size': 512, 'dropout': 0.4, 'hidden': 64, 'epochs': 30},
    {'name': 'lr_high',     'lr': 0.003, 'batch_size': 128, 'dropout': 0.2, 'hidden': 32, 'epochs': 30},
    {'name': 'small_model', 'lr': 0.001, 'batch_size': 256, 'dropout': 0.3, 'hidden': 16, 'epochs': 30},
]

N_CONFIGS = len(CONFIGS)
print(f'Configs: {N_CONFIGS}')

# Paths
DATA_DIR = Path('/kaggle/input/displacement-data')
OUT_DIR = Path('/kaggle/working/displacement_results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

M5_PQ = DATA_DIR / 'displacement_dataset_v1.parquet'
HTF_PQ = DATA_DIR / 'm5_htf_context_v1.parquet'
M5_RAW_PQ = DATA_DIR / 'EURUSD_M5.parquet'  # dataset crudo para reconstruir 'time'

# Si no existen en Kaggle, usar paths locales
if not M5_PQ.exists():
    M5_PQ = Path('data/learning/pipeline/displacement/displacement_dataset_v1.parquet')
    HTF_PQ = Path('data/learning/pipeline/displacement/m5_htf_context_v1.parquet')

assert M5_PQ.exists(), f'Falta: {M5_PQ}'
assert HTF_PQ.exists(), f'Falta: {HTF_PQ}'

# Features (idénticas al bloque 4)
BASE_F = [
    'feature_body_range_ratio','feature_body_pips','feature_range_pips',
    'feature_upper_wick','feature_lower_wick','feature_wick_ratio',
    'feature_close_vs_open','feature_close_vs_prev_close',
    'feature_volatility_5','feature_swing_change',
    'feature_sweep_previo','feature_fvg_cercano','feature_ob_cercano',
    'feature_struct_confirmed','feature_fvg_count_5','feature_ob_present',
    'hist_feature_body_range_ratio_mean','hist_feature_body_range_ratio_std',
    'hist_feature_wick_ratio_mean','hist_feature_wick_ratio_std',
    'hist_feature_close_vs_open_mean','hist_feature_close_vs_open_std',
    'hist_body_ratio_max','hist_range_ratio_avg'
]

HTF_F = ['h1_sesgo','h1_fuerza','h1_range','h4_sesgo','h4_fuerza','h4_range','d1_sesgo','d1_fuerza','d1_range']
ALL_F = BASE_F + HTF_F

print(f'Features: {len(ALL_F)} (base {len(BASE_F)} + HTF {len(HTF_F)})')

In [ ]:
# === CARGA DE DATOS (una sola vez) ===
print('=== Cargando datos ===')
t0 = time.time()

# 1. M5 etiquetado: tiene columna 'idx' (int64), NO tiene 'time'
m5_ds = pd.read_parquet(M5_PQ)
print(f'M5 etiquetado: {len(m5_ds)} filas' + f', cols: {m5_ds.columns.tolist()[:5]}...' if len(m5_ds.columns) > 5 else f'M5 etiquetado: {len(m5_ds)} filas')
assert 'idx' in m5_ds.columns, 'El M5 etiquetado debe tener columna idx'

# 2. M5 crudo: tiene 'time' (datetime64[us, UTC]), es el lookup table
m5_raw = pd.read_parquet(M5_RAW_PQ)
print(f'M5 crudo: {len(m5_raw)} filas, time dtype: {m5_raw["time"].dtype}')

# 3. M5 HTF context: tiene 'time' (datetime64[us])
m5_htf = pd.read_parquet(HTF_PQ)
print(f'M5 HTF: {len(m5_htf)} filas, time dtype: {m5_htf["time"].dtype}')

print(f'Tiempo carga: {time.time()-t0:.1f}s')

# === RECUPERAR 'time' EN M5 ETIQUETADO USANDO 'idx' ===
# idx contiene posiciones (0-based) en el M5 crudo
print('\nRecuperando time desde M5 crudo usando idx...')

# Asegurar que m5_raw está ordenado por posición (0,1,2,...)
m5_raw = m5_raw.reset_index(drop=True)

# Extraer time usando idx como índice posicional:
m5_ds['time'] = m5_raw['time'].iloc[m5_ds['idx'].values].values

# Eliminar timezone (raw tiene UTC, HTF no tiene) para merge_asof
if hasattr(m5_ds['time'].dtype, 'tz') and m5_ds['time'].dtype.tz is not None:
    m5_ds['time'] = m5_ds['time'].dt.tz_localize(None)

# Ordenar ambos por time (merge_asof requiere sorted input)
m5_ds = m5_ds.sort_values('time').reset_index(drop=True)
m5_htf = m5_htf.sort_values('time').reset_index(drop=True)

# Convertir time a int64 (nanosecondes desde epoch) para merge_asof
m5_ds['time_int'] = m5_ds['time'].astype('int64')
m5_htf['time_int'] = m5_htf['time'].astype('int64')

print(f'M5 etiquetado time rango: {m5_ds["time"].min()} -> {m5_ds["time"].max()}')
print(f'M5 HTF time rango: {m5_htf["time"].min()} -> {m5_htf["time"].max()}')
print(f'time_int dtype: {m5_ds["time_int"].dtype}')

# === MERGE ASOF: alinear M5 etiquetado con contexto HTF ===
print('\nAlineando M5 con contexto HTF via merge_asof...')
start_merge = time.time()

ds = pd.merge_asof(
    m5_ds,
    m5_htf,
    left_on='time_int',
    right_on='time_int',
    direction='backward',
    allow_exact_matches=True
)

print(f'Merge: {len(ds)} filas ({time.time()-start_merge:.1f}s)')

# Rellenar NaN en HTF features (primeras velas sin contexto HTF)
for col in HTF_F:
    if col in ds.columns:
        n_nan = ds[col].isna().sum()
        if n_nan > 0:
            ds[col] = ds[col].fillna(0.0)
            print(f'  {col}: {n_nan} NaN rellenados con 0')
    else:
        ds[col] = 0.0
        print(f'  {col}: columna missing, creada con 0')

# Targets
ds['target_geo'] = ds['geometric_strength'].map({'NONE':0,'WEAK':1,'STRONG':2}).values
ds['target_dir'] = ds['direction'].map({'NONE':0,'UP':1,'DOWN':2}).values
ds['target_ict'] = ds['ict_context_status'].map({'NOT_SUPPORTED':0,'SUPPORTED':1,'UNKNOWN':2}).values
ds['target_usable'] = (
    (ds['geometric_strength'] != 'NONE') &
    (ds['direction'] != 'NONE') &
    (ds['ict_context_status'] == 'SUPPORTED') &
    (ds['episode_status'] == 'CONFIRMED')
).astype(int).values

# Features de entrada
for f in ALL_F:
    if f not in ds.columns:
        ds[f] = 0.0

X = ds[ALL_F].values.astype(np.float32)
y_geo = ds['target_geo'].values.astype(np.int64)
y_dir = ds['target_dir'].values.astype(np.int64)
y_ict = ds['target_ict'].values.astype(np.int64)
y_usable = ds['target_usable'].values.astype(np.int64)

print(f'\nDatos listos: X={X.shape}')
print(f'y_geo: {np.bincount(y_geo)}')
print(f'y_dir: {np.bincount(y_dir)}')
print(f'y_ict: {np.bincount(y_ict)}')
print(f'y_usable: {y_usable.sum()} positivos / {len(y_usable)} total')

# Splits temporales 70/15/15
n = len(X)
split_t = int(0.70 * n)
split_v = int(0.85 * n)

X_tr, X_va, X_te = X[:split_t], X[split_t:split_v], X[split_v:]
y_tr_g, y_va_g, y_te_g = y_geo[:split_t], y_geo[split_t:split_v], y_geo[split_v:]
y_tr_d, y_va_d, y_te_d = y_dir[:split_t], y_dir[split_t:split_v], y_dir[split_v:]
y_tr_i, y_va_i, y_te_i = y_ict[:split_t], y_ict[split_t:split_v], y_ict[split_v:]
y_tr_u, y_va_u, y_te_u = y_usable[:split_t], y_usable[split_t:split_v], y_usable[split_v:]

print(f'\nSplit: TRAIN={len(X_tr)}, VAL={len(X_va)}, TEST={len(X_te)}')
print(f'USABLE en TEST: {y_te_u.sum()}')

# Normalizacion
mean = X_tr.mean(axis=0, keepdims=True)
std = np.where(X_tr.std(axis=0, keepdims=True) == 0, 1.0, X_tr.std(axis=0, keepdims=True))
X_tr_n = (X_tr - mean) / std
X_va_n = (X_va - mean) / std
X_te_n = (X_te - mean) / std

# Para GRU: dimension temporal
X_tr_g = X_tr_n.reshape(-1, 1, X_tr_n.shape[1])
X_va_g = X_va_n.reshape(-1, 1, X_va_n.shape[1])
X_te_g = X_te_n.reshape(-1, 1, X_te_n.shape[1])

print(f'X_train shape para GRU: {X_tr_g.shape}')


In [ ]:
# === MODELO GRU ===
def build_gru(n_features, hidden, dropout, n_classes):
    model = keras.Sequential([
        layers.GRU(hidden, input_shape=(1, n_features), batch_size=None),
        layers.Dense(hidden, activation='relu'),
        layers.Dropout(dropout),
        layers.Dense(n_classes, activation='softmax')
    ])
    return model

def train_one_config(args):
    '''
    Función standalone para multiprocessing.
    Entrena una configuración y retorna resultados serializables.
    Usa tf.keras.backend.clear_session() para evitar memory leaks en multiprocessing.
    '''
    (cfg, target_name, y_train, y_val, y_test, n_classes,
     X_tr, X_va, X_te, worker_id) = args
    
    # Reiniciar TF en cada worker para evitar memory leaks en multiprocessing
    tf.keras.backend.clear_session()
    
    print(f'  [Worker {worker_id}] {cfg["name"]} | {target_name}...')
    t0 = time.time()
    
    model = build_gru(X_tr.shape[2], cfg['hidden'], cfg['dropout'], n_classes)
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=cfg['lr']),
        loss='sparse_categorical_crossentropy',
        metrics=['accuracy']
    )
    
    # Early stopping (VIGIL's recommendation)
    early_stop = keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=0
    )
    
    history = model.fit(
        X_tr, y_train,
        validation_data=(X_va, y_val),
        epochs=cfg['epochs'],
        batch_size=cfg['batch_size'],
        callbacks=[early_stop],
        verbose=0
    )
    
    pred_te = model.predict(X_te, verbose=0).argmax(axis=1)
    acc = accuracy_score(y_test, pred_te)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_test, pred_te, labels=list(range(n_classes)), average=None, zero_division=0
    )
    
    elapsed = time.time() - t0
    
    return {
        'config': cfg['name'],
        'target': target_name,
        'acc_test': float(acc),
        'time_seconds': float(elapsed),
        'epochs_run': len(history.history['loss']),
        'final_val_loss': float(history.history['val_loss'][-1]),
        'precisions': [float(p) for p in prec],
        'recalls': [float(r) for r in rec],
        'f1s': [float(f) for f in f1],
        'worker_id': worker_id,
    }

In [ ]:
# === ENTRENAMIENTO CON MULTIPROCESSING (CPU paralelo ≥10 núcleos) ===
# Usa multiprocessing para ejecutar múltiples configs en paralelo
# Cada worker es un proceso separado con su propia sesión TF

print('='*60)
print(f'FASE 1: Corrida rápida de {N_CONFIGS} configuraciones')
print(f'Workers: {NUM_WORKERS} CPUs en paralelo (mínimo 10)')
print('='*60)

all_targets = [
    ('geo', y_tr_g, y_va_g, y_te_g, 3),
    ('dir', y_tr_d, y_va_d, y_te_d, 3),
    ('ict', y_tr_i, y_va_i, y_te_i, 2),
    ('usable', y_tr_u, y_va_u, y_te_u, 2),
]

results_fase1 = []
worker_counter = 0

for target_name, y_tr, y_va, y_te, n_classes in all_targets:
    print(f'\n=== Target: {target_name} ({n_classes} clases) ===')
    
    if target_name == 'usable' and y_te.sum() < 10:
        print(f'  ⚠️  USABLE: solo {y_te.sum()} positivos en test')
    
    # Preparar tareas para multiprocessing
    tasks = [
        (cfg, target_name, y_tr, y_va, y_te, n_classes,
         X_tr_g, X_va_g, X_te_g, worker_counter + i)
        for i, cfg in enumerate(CONFIGS)
    ]
    worker_counter += len(CONFIGS)
    
    t_start = time.time()
    
    # Ejecutar en paralelo con multiprocessing
    # Usar NUM_WORKERS procesos simultáneos
    with mp.Pool(processes=NUM_WORKERS) as pool:
        batch_results = pool.map(train_one_config, tasks)
    
    results_fase1.extend(batch_results)
    
    elapsed = time.time() - t_start
    print(f'  ⏱ {len(tasks)} configs en {elapsed:.1f}s ({elapsed/len(tasks):.1f}s por config en promedio)')
    
    # Ranking
    target_results = [r for r in results_fase1 if r['target'] == target_name]
    sorted_r = sorted(target_results, key=lambda x: x['acc_test'], reverse=True)
    print(f'  Ranking {target_name}:')
    for i, r in enumerate(sorted_r):
        print(f'    {i+1}. [{r["config"]}] acc={r["acc_test"]:.4f}, time={r["time_seconds"]:.1f}s, epochs={r["epochs_run"]}, worker={r["worker_id"]}')

print(f'\n✅ FASE 1 completada: {len(results_fase1)} resultados en total')

In [ ]:
# === FASE 2: Reentrenar top configs con más epochs ===
# Los mejores configs de cada target reciben más epochs (50 en lugar de 30)
print('\n' + '='*60)
print('FASE 2: Reentrenar top configs con más epochs')
print('='*60)

results_fase2 = []

for target_name, y_tr, y_va, y_te, n_classes in all_targets:
    print(f'\n=== Target: {target_name} ===')
    
    target_results = [r for r in results_fase1 if r['target'] == target_name]
    if not target_results:
        continue
    
    top3 = sorted(target_results, key=lambda x: x['acc_test'], reverse=True)[:3]
    
    print(f'  Top 3 configs:')
    for r in top3:
        print(f'    - [{r["config"]}] acc={r["acc_test"]:.4f}')
    
    tasks_fase2 = []
    for r in top3:
        cfg = next(c for c in CONFIGS if c['name'] == r['config'])
        cfg2 = dict(cfg)
        cfg2['epochs'] = 50
        tasks_fase2.append((cfg2, target_name, y_tr, y_va, y_te, n_classes,
                           X_tr_g, X_va_g, X_te_g, worker_counter))
        worker_counter += 1
    
    t_start = time.time()
    with mp.Pool(processes=NUM_WORKERS) as pool:
        batch_results = pool.map(train_one_config, tasks_fase2)
    
    results_fase2.extend(batch_results)
    print(f'  ⏱ {len(tasks_fase2)} configs reentrenados en {time.time()-t_start:.1f}s')

In [ ]:
# === RESUMEN FINAL ===
print('\n' + '='*60)
print('RESUMEN FINAL — Todos los resultados')
print('='*60)

all_results = results_fase1 + results_fase2
total_time = sum(r['time_seconds'] for r in all_results)

print(f'\nTotal corridas: {len(all_results)}')
print(f'Tiempo total de entrenamiento: {total_time:.1f}s')
print(f'Workers usados: {NUM_WORKERS} CPUs')
print(f'GPU disponible: {"Sí" if len(gpus) > 0 else "No"}')

for target_name in ['geo','dir','ict','usable']:
    target_results = [r for r in all_results if r['target'] == target_name]
    if not target_results:
        continue
    
    print(f'\n{target_name.upper()} (n={len(target_results)} corridas):')
    top5 = sorted(target_results, key=lambda x: x['acc_test'], reverse=True)[:5]
    for i, r in enumerate(top5):
        print(f'  {i+1}. [{r["config"]}] acc={r["acc_test"]:.4f}, time={r["time_seconds"]:.1f}s, epochs={r["epochs_run"]}, f1_mean={np.mean(r["f1s"]):.4f}')

print('\n--- Configuración ganadora por target ---')
for target_name in ['geo','dir','ict','usable']:
    target_results = [r for r in all_results if r['target'] == target_name]
    if not target_results:
        continue
    best = max(target_results, key=lambda x: x['acc_test'])
    print(f'  {target_name}: {best["config"]} → acc={best["acc_test"]:.4f}')

# Guardar resultados
summary = {
    'phase1': results_fase1,
    'phase2': results_fase2,
    'configs_used': [c['name'] for c in CONFIGS],
    'features': ALL_F,
    'total_time_seconds': total_time,
    'workers_used': NUM_WORKERS,
    'gpu_available': len(gpus) > 0,
    'note': f'{NUM_WORKERS} CPUs paralelos + GPU={len(gpus)}'
}

with open(OUT_DIR / 'results.json', 'w') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print(f'\n✓ Resultados guardados: {OUT_DIR / "results.json"}')
print(f'  Workers: {NUM_WORKERS} CPUs')
print(f'  GPU: {"Sí" if len(gpus) > 0 else "No"}')
print(f'  Total tiempo: {total_time:.1f}s')

print('\n' + '='*60)
print('SENTINEL NOTE:')
print('Resultados son preliminares. Validar con test hold-out')
print('antes de concluir que una config es superior.')
print('='*60)

---

## Instrucciones para Kaggle

### 1. Subir datos
- Ve a **kaggle.com → Datasets → New Dataset**
- Sube:
  - `data/learning/pipeline/displacement/displacement_dataset_v1.parquet`
  - `data/learning/pipeline/displacement/m5_htf_context_v1.parquet`
- Nombre: **displacement-data**

### 2. Subir notebook
- **kaggle.com → Notebooks → New Notebook → Upload**
- Seleccioná `kaggle_notebook_displacement.ipynb`

### 3. Habilitar GPU
- **Settings → Accelerator → GPU T4 x2**
- **Save**

### 4. Ejecutar
- **Connect** (arriba a la derecha)
- **Run All**

### 5. Descargar resultados
- Ve a **Output** → descargá `displacement_results/results.json`

---

## Uso de CPU

- Detecta automáticamente `mp.cpu_count()` CPUs disponibles
- Usa **al menos 10 CPUs** como mínimo (si hay menos, usa todos)
- Con GPU: usa CPU para data loading y coordinación
- Sin GPU: usa CPU paralelo con multiprocessing para todos los entrenamientos
- Cada worker corre una config independiente en paralelo
- `tf.keras.backend.clear_session()` en cada worker para evitar memory leaks

---

## Agentes que diseñaron esto

| Agente | Contribución |
|--------|-------------|
| **FORGE** | Infraestructura GPU + CPU paralelo |
| **PROBE** | 8 configs en lugar de 32 (diseño válido) |
| **SENTINEL** | Validación estadística + disclaimer |
| **VIGIL** | Early-stop + monitoreo en tiempo real |
| **NEXUS** | Coordinación + poda adaptativa |

---